In [5]:
import pandas as pd
import numpy as np
import gc
import os
import warnings
warnings.filterwarnings('ignore')

RAW_DATA_PATH = "../data/Jan - May '26 Data.csv"
OUTPUT_PATH   = '../processed/retailer_day_features.parquet'

os.makedirs('../processed', exist_ok=True)
print('Setup complete.')

Setup complete.


In [6]:
print('Loading raw data...')
df = pd.read_csv(RAW_DATA_PATH)
df['createdAt'] = pd.to_datetime(df['createdAt'], dayfirst=True)
print(f'Raw rows loaded: {len(df):,}')

# Keep only confirmed orders
confirmed = df[df['orderStatus'].isin(['Delivered', 'PartiallyDelivered'])].copy()
print(f'Confirmed SKU rows: {len(confirmed):,}')

# Deduplicate to one row per order
# Keep static retailer info alongside
orders = confirmed.drop_duplicates(subset='orderNumber')[[
    'orderNumber', 'customerId', 'createdAt',
    'hubName', 'shopType', 'retailerType', 'orderSource'
]].copy()

print(f'Unique confirmed orders: {len(orders):,}')
print(f'Unique retailers: {orders["customerId"].nunique():,}')

# Free memory
del df, confirmed
gc.collect()
print('Memory freed.')

Loading raw data...
Raw rows loaded: 609,723
Confirmed SKU rows: 543,130
Unique confirmed orders: 177,340
Unique retailers: 8,640
Memory freed.


In [7]:
# Get most common value per retailer for static attributes
retailer_profile = orders.groupby('customerId').agg(
    hubName      = ('hubName',      lambda x: x.mode()[0]),
    shopType     = ('shopType',     lambda x: x.mode()[0]),
    retailerType = ('retailerType', lambda x: x.mode()[0] if x.notna().any() else 'Unknown'),
    app_orders   = ('orderSource',  lambda x: (x == 'App').sum()),
    call_orders  = ('orderSource',  lambda x: (x == 'CALLING_AGENT').sum()),
    total_orders = ('orderNumber',  'count')
).reset_index()

# App usage ratio — retailers who mostly use App need less calling
retailer_profile['app_order_ratio'] = (
    retailer_profile['app_orders'] / retailer_profile['total_orders']
).round(4)

print('Retailer profile shape:', retailer_profile.shape)
retailer_profile.head(3)

Retailer profile shape: (8640, 8)


,customerId,hubName,shopType,retailerType,app_orders,call_orders,total_orders,app_order_ratio
0,USR-100,Crossline Events (Noida),Paan A,HVLF,12,1,13,0.9231
1,USR-1000,Instant Foods(Noida),General B,HVLF,0,3,3,0.0000
2,USR-100021,Instant Foods (SED),General A,HVLF,2,2,4,0.5000


In [8]:
all_retailers = orders['customerId'].unique()
all_dates = pd.date_range(start=orders['createdAt'].min(),
                          end=orders['createdAt'].max(),
                          freq='D')

print(f'Retailers: {len(all_retailers):,}')
print(f'Days: {len(all_dates)}')
print(f'Grid size: {len(all_retailers) * len(all_dates):,} rows')
print('Building grid (may take 1-2 min)...')

# Build cross join using MultiIndex — memory efficient
idx = pd.MultiIndex.from_product([all_retailers, all_dates],
                                  names=['customerId', 'date'])
grid = pd.DataFrame(index=idx).reset_index()

print(f'Grid built: {len(grid):,} rows')

Retailers: 8,640
Days: 151
Grid size: 1,304,640 rows
Building grid (may take 1-2 min)...
Grid built: 1,304,640 rows


In [9]:
# Create order flags at retailer-day level
order_flags = orders[['customerId', 'createdAt']].copy()
order_flags = order_flags.rename(columns={'createdAt': 'date'})
order_flags['ordered_today'] = 1

# Handle same retailer ordering multiple times in one day (count as 1)
order_flags = order_flags.drop_duplicates(subset=['customerId', 'date'])

# Merge into grid
grid = grid.merge(order_flags, on=['customerId', 'date'], how='left')
grid['ordered_today'] = grid['ordered_today'].fillna(0).astype(int)

print(f'Total rows: {len(grid):,}')
print(f'Positive (ordered=1): {grid["ordered_today"].sum():,} ({100*grid["ordered_today"].mean():.1f}%)')
print(f'Negative (ordered=0): {(grid["ordered_today"]==0).sum():,}')

Total rows: 1,304,640
Positive (ordered=1): 166,864 (12.8%)
Negative (ordered=0): 1,137,776


In [10]:
print('Sorting grid...')
grid = grid.sort_values(['customerId', 'date']).reset_index(drop=True)

# Create a binary order column for rolling calculations
grid['_ordered'] = grid['ordered_today']

print('Computing rolling order counts...')

# Rolling order counts (shift by 1 so we exclude today — no leakage)
grouped = grid.groupby('customerId')['_ordered']

grid['orders_last_7_days']  = grouped.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).sum())
grid['orders_last_14_days'] = grouped.transform(lambda x: x.shift(1).rolling(14, min_periods=1).sum())
grid['orders_last_30_days'] = grouped.transform(lambda x: x.shift(1).rolling(30, min_periods=1).sum())
grid['total_orders_so_far'] = grouped.transform(lambda x: x.shift(1).expanding().sum())

print('Rolling counts done.')

Sorting grid...
Computing rolling order counts...
Rolling counts done.


In [11]:
print('Computing days since last order...')

# For each row, find the date of the most recent order before today
# Strategy: where ordered=0, propagate last known order date forward
grid['_order_date'] = grid['date'].where(grid['_ordered'] == 1)
grid['last_order_date'] = grid.groupby('customerId')['_order_date'].transform(
    lambda x: x.shift(1).ffill()
)

grid['days_since_last_order'] = (grid['date'] - grid['last_order_date']).dt.days

# Fill NaN (retailers with no prior order) with a large number
grid['days_since_last_order'] = grid['days_since_last_order'].fillna(999)

print('days_since_last_order done.')

Computing days since last order...
days_since_last_order done.


In [12]:
print('Computing inter-order gap statistics...')

# Build gap stats per retailer from the orders dataframe
orders_sorted = orders.sort_values(['customerId', 'createdAt'])
orders_sorted['gap'] = orders_sorted.groupby('customerId')['createdAt'].diff().dt.days

gap_stats = orders_sorted.groupby('customerId')['gap'].agg(
    avg_gap_between_orders = 'mean',
    std_gap_between_orders = 'std',
    min_gap                = 'min',
    max_gap                = 'max'
).reset_index()

gap_stats['avg_gap_between_orders'] = gap_stats['avg_gap_between_orders'].fillna(30).round(2)
gap_stats['std_gap_between_orders'] = gap_stats['std_gap_between_orders'].fillna(0).round(2)
gap_stats['min_gap'] = gap_stats['min_gap'].fillna(0)
gap_stats['max_gap'] = gap_stats['max_gap'].fillna(0)

# Merge into grid
grid = grid.merge(gap_stats, on='customerId', how='left')

print('Gap stats done.')

Computing inter-order gap statistics...
Gap stats done.


In [13]:
print('Computing days_overdue and order_regularity...')

# days_overdue: how many days past expected next order
# Expected next order = last_order_date + avg_gap
grid['expected_next_order'] = grid['last_order_date'] + pd.to_timedelta(grid['avg_gap_between_orders'], unit='D')
grid['days_overdue'] = (grid['date'] - grid['expected_next_order']).dt.days
grid['days_overdue'] = grid['days_overdue'].fillna(0)

# is_overdue flag
grid['is_overdue'] = (grid['days_overdue'] > 0).astype(int)

# Order regularity: lower std = more regular = more predictable
# Avoid division by zero
grid['order_regularity'] = 1 / (grid['std_gap_between_orders'] + 1)

print('Overdue features done.')

Computing days_overdue and order_regularity...
Overdue features done.


In [14]:
print('Adding temporal features...')

grid['day_of_week']   = grid['date'].dt.dayofweek   # 0=Monday, 6=Sunday
grid['day_of_month']  = grid['date'].dt.day
grid['week_of_month'] = (grid['date'].dt.day - 1) // 7 + 1
grid['month']         = grid['date'].dt.month
grid['is_weekend']    = (grid['date'].dt.dayofweek >= 5).astype(int)
grid['is_month_start']= (grid['date'].dt.day <= 3).astype(int)   # first 3 days
grid['is_month_end']  = (grid['date'].dt.day >= 28).astype(int)  # last ~3 days

print('Temporal features done.')

Adding temporal features...
Temporal features done.


In [15]:
print('Merging retailer profile...')
grid = grid.merge(
    retailer_profile[['customerId','hubName','shopType','retailerType','app_order_ratio']],
    on='customerId', how='left'
)

# Fill missing retailerType
grid['retailerType'] = grid['retailerType'].fillna('Unknown')

print('Profile merged.')
print(f'Grid shape after all features: {grid.shape}')

Merging retailer profile...
Profile merged.
Grid shape after all features: (1304640, 30)


In [16]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['hubName', 'shopType', 'retailerType']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    grid[col + '_enc'] = le.fit_transform(grid[col].astype(str))
    label_encoders[col] = le
    print(f'{col}: {grid[col].nunique()} unique values encoded')

print('Encoding complete.')

hubName: 11 unique values encoded
shopType: 10 unique values encoded
retailerType: 4 unique values encoded
Encoding complete.


In [17]:
# Drop intermediate helper columns
drop_cols = ['_ordered', '_order_date', 'last_order_date', 'expected_next_order',
             'min_gap', 'max_gap']
grid = grid.drop(columns=[c for c in drop_cols if c in grid.columns])

# Final feature list
FEATURE_COLS = [
    'days_since_last_order',
    'avg_gap_between_orders',
    'std_gap_between_orders',
    'orders_last_7_days',
    'orders_last_14_days',
    'orders_last_30_days',
    'total_orders_so_far',
    'days_overdue',
    'is_overdue',
    'order_regularity',
    'app_order_ratio',
    'day_of_week',
    'day_of_month',
    'week_of_month',
    'month',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'hubName_enc',
    'shopType_enc',
    'retailerType_enc',
]

TARGET_COL = 'ordered_today'

print('Final feature columns:')
for f in FEATURE_COLS:
    print(f'  {f}')

print(f'\nTarget: {TARGET_COL}')
print(f'Final grid shape: {grid.shape}')

Final feature columns:
  days_since_last_order
  avg_gap_between_orders
  std_gap_between_orders
  orders_last_7_days
  orders_last_14_days
  orders_last_30_days
  total_orders_so_far
  days_overdue
  is_overdue
  order_regularity
  app_order_ratio
  day_of_week
  day_of_month
  week_of_month
  month
  is_weekend
  is_month_start
  is_month_end
  hubName_enc
  shopType_enc
  retailerType_enc

Target: ordered_today
Final grid shape: (1304640, 27)


In [18]:
# Check for any remaining nulls in feature columns
null_check = grid[FEATURE_COLS].isnull().sum()
print('=== NULL CHECK IN FEATURES ===')
print(null_check[null_check > 0])
if null_check.sum() == 0:
    print('No nulls. Clean.')

=== NULL CHECK IN FEATURES ===
orders_last_7_days     8640
orders_last_14_days    8640
orders_last_30_days    8640
total_orders_so_far    8640
dtype: int64


In [19]:
# Check for any remaining nulls in feature columns
null_check = grid[FEATURE_COLS].isnull().sum()
print('=== NULL CHECK IN FEATURES ===')
print(null_check[null_check > 0])
if null_check.sum() == 0:
    print('No nulls. Clean.')

=== NULL CHECK IN FEATURES ===
orders_last_7_days     8640
orders_last_14_days    8640
orders_last_30_days    8640
total_orders_so_far    8640
dtype: int64


In [20]:
print('Saving to parquet...')
grid.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved to: {OUTPUT_PATH}')
print(f'File size: {os.path.getsize(OUTPUT_PATH) / (1024**2):.1f} MB')
print()
print('=== FEATURE ENGINEERING COMPLETE ===')
print(f'Final dataset: {len(grid):,} rows × {len(grid.columns)} columns')
print(f'Positive samples: {grid[TARGET_COL].sum():,} ({100*grid[TARGET_COL].mean():.1f}%)')
print()
print('Next: Run Notebook 3 (Model Training)')

Saving to parquet...
Saved to: ../processed/retailer_day_features.parquet
File size: 3.0 MB

=== FEATURE ENGINEERING COMPLETE ===
Final dataset: 1,304,640 rows × 27 columns
Positive samples: 166,864 (12.8%)

Next: Run Notebook 3 (Model Training)
